# Training Pipeline YOLOv8 & CrossKD (RDD2022)
Tugas Akhir: *Pengembangan Model Lightweight YOLOv8 untuk Deteksi Kerusakan Jalan Menggunakan Cross-Head Knowledge Distillation*

## 1. Cek Ketersediaan GPU

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Tersedia : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

possible_paths = [
    Path("/content/drive/MyDrive/Iqbal/TA CROSSKD"),
    Path("/content/drive/MyDrive/TA CROSSKD"),
    Path("/content/drive/Shareddrives/TA CROSSKD"),
]

DRIVE_DIR = next((p for p in possible_paths if p.exists()), None)

if DRIVE_DIR:
    print(f"Folder ditemukan: {DRIVE_DIR}")
    print("Isi folder:", [f.name for f in DRIVE_DIR.iterdir()])
else:
    print("Folder 'TA CROSSKD' tidak ditemukan, periksa path Google Drive.")

## 3. Ekstrak Kode & Dataset ke Disk Lokal Colab

In [ ]:
from pathlib import Path

code_zip = DRIVE_DIR / "code_lkd_fix.zip"
if not Path("/content/ultralytics").exists() and code_zip.exists():
    !unzip -q "{code_zip}" -d /content/
    print("Kode diekstrak.")

dataset_zip = DRIVE_DIR / "dataset_rdd2022_augmented.zip"
if not Path("/content/dataset_rdd2022_augmented/images").exists() and dataset_zip.exists():
    !unzip -q "{dataset_zip}" -d /content/
    print("Dataset diekstrak.")

## 4. Konfigurasi `data.yaml` & Dependensi

In [ ]:
import re
from pathlib import Path

!pip uninstall -y -q wandb
!pip install -q opencv-python-headless matplotlib pyyaml scipy tqdm psutil pandas seaborn thop

yaml_path = Path("/content/dataset_rdd2022_augmented/data.yaml")
if yaml_path.exists():
    content = yaml_path.read_text(encoding="utf-8")
    content = re.sub(r"path:.*", "path: /content/dataset_rdd2022_augmented", content)
    yaml_path.write_text(content, encoding="utf-8")
    print(content)
else:
    print("data.yaml tidak ditemukan.")

## 5. Training Model Teacher (YOLOv8m)

In [ ]:
import os
import sys

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

sys.path.insert(0, "/content")
os.chdir("/content")

if DRIVE_DIR:
    drive_detect = DRIVE_DIR / "runs" / "detect"
    drive_detect.mkdir(parents=True, exist_ok=True)
    !rm -rf /content/TA_PROJECT
    !ln -s "{drive_detect}" /content/TA_PROJECT

from ultralytics import YOLO, settings
settings.update({"wandb": False})

try:
    import ultralytics.utils.callbacks.wb as wb_cb
    wb_cb.callbacks.clear()
except Exception:
    pass

model = YOLO("/content/PretrainedModelFromOfficial/yolov8m.pt")
model.train(
    data="/content/dataset_rdd2022_augmented/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    workers=2,
    device=0,
    pretrained=True,
    iou=0.6,
    patience=20,
    cos_lr=True,
    close_mosaic=15,
    weight_decay=0.001,
    project="TA_PROJECT",
    name="teacher_yolov8m_optimized",
    exist_ok=True,
    Mode="Training",
    quant=False,
    amp=True,
)

## 6. Resume Training (Jika Terputus)

In [ ]:
from ultralytics import YOLO

last_ckpt = DRIVE_DIR / "runs" / "detect" / "teacher_yolov8m_optimized" / "weights" / "last.pt"

if last_ckpt.exists():
    model_resume = YOLO(str(last_ckpt))
    model_resume.train(resume=True)
else:
    print(f"Checkpoint '{last_ckpt}' tidak ditemukan.")

## 7. Visualisasi Metrik Evaluasi

In [ ]:
from IPython.display import Image, display

res_dir = DRIVE_DIR / "runs" / "detect" / "teacher_yolov8m_optimized"

for p in ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    img_path = res_dir / p
    if img_path.exists():
        print(p)
        display(Image(filename=str(img_path)))
    else:
        print(f"File {p} belum tersedia.")